In [1]:
import pandas as pd
import glob
import os

## Step 0 

In [4]:
# Notice the 'r' before the string, and the single backslashes
df_201605 = pd.read_excel(r'2016.05 방한외래관광객 세부통계.xlsx', sheet_name=1, header=0) 

In [3]:
# ==========================================
# STEP 1: Translate the Column Headers
# ==========================================
column_translation = {
    '년월': 'Month',
    '목적': 'Purpose',
    '연령': 'Age',
    '성별': 'Gender',
    '국적': 'Nationality',
    '지역': 'Region',
    '입국항': 'Transport',
    '교통수단': 'Mode of Transport',
    '인원': 'Number'
}
df_201605 = df_201605.rename(columns=column_translation)
# -> THE FIX: Force the date to match the English format BEFORE merging
date= '2016-05'
df_201605['Month'] = date

In [4]:
# ==========================================
# 3. Strip Hidden Spaces (Vectorized)
# ==========================================
# This cleans '일 본' to '일본', '미 주' to '미주', '제 주' to '제주' all at once!
cols_with_spaces = ['Nationality', 'Region', 'Transport']
for col in cols_with_spaces:
    df_201605[col] = df_201605[col].astype(str).str.replace(' ', '')

In [5]:
# ==========================================
# STEP 2: Translate the Categorical Values
# ==========================================
# We create dictionaries for the core categories
purpose_map = {
    '관광': 'Tour',
    '상용': 'Business',
    '공용': 'Official',
    '유학연수': 'Training of Study Abroad',
    '기타': 'Others'
}

gender_map = {
    '여성': 'Female',
    '남성': 'Male',
    '승무원': 'Crew'
}

age_map = {
    '승무원': 'Crew' # Fixes the Crew age bug
} 

mode_map = {
    '공항': 'Airport',
    '항구': 'Port',
    '기타': 'Others'
}

region_map = {
    '아시아주': 'Asia and the Pacific',
    '미주': 'Americas',
    '구주': 'Europe',
    '대양주': 'Oceania',
    '아프리카주': 'Africa',
    '중동': 'Middle East',
    '기타': 'Others',
    '교포': 'Others'
}

transport_map = {
    '인천공항': 'Incheon Airport', '김해': 'Gimhae', '김포공항': 'Gimpo Airport', '제주': 'Jeju',
    '제주공항': 'Jeju Airport', '부산': 'Pusan', '인천': 'Incheon', '대구': 'Daegu',
    '청주': 'Cheongju', '광주': 'Gwangju', '무안': 'Muan', '양양공항': 'Yangyang Airport',
    '평택': 'Pyeongtaek', '광양': 'Gwangyang', '군산': 'Gunsan', '동해': 'Donghae',
    '속초': 'Sokcho', '마산': 'Masan', '사천': 'Sacheon', '목포': 'Mokpo', '여수': 'Yeosu',
    '포항': 'Pohang', '울산': 'Ulsan', '통영': 'Tongyeong', '거제': 'Geoje', '대산': 'Daesan',
    '오산': 'Osan', '성남': 'Seongnam', '감천': 'Gamcheon', '당진': 'Dangjin', '기타': 'Others'
}

nation_map = {
    '일본': 'JAPAN', '중국': 'CHINA P. R.', '대만': 'TAIWAN', '홍콩': 'HONG KONG, CHINA', '마카오': 'MACAO',
    '필리핀': 'PHILIPPINES', '태국': 'THAILAND', '말레이시아': 'MALAYSIA', '싱가포르': 'SINGAPORE',
    '인도네시아': 'INDONESIA', '베트남': 'VIETNAM', '인도': 'INDIA', '미얀마': 'MYANMAR', '스리랑카': 'SRI LANKA',
    '방글라데시': 'BANGLADESH', '파키스탄': 'PAKISTAN', '몽골': 'MONGOLIA', '캄보디아': 'CAMBODIA',
    '우즈베키스탄': 'UZBEKISTAN', '카자흐스탄': 'KAZAKHSTAN', '네팔': 'NEPAL', '아시아주기타': 'Asia Others',
    '미국': 'UNITED STATES', '캐나다': 'CANADA', '멕시코': 'MEXICO', '브라질': 'BRAZIL', '아르헨티나': 'ARGENTINA',
    '칠레': 'CHILE', '콜롬비아': 'COLOMBIA', '페루': 'PERU', '미주기타': 'Americas Others',
    '러시아': 'RUSSIA', '영국': 'UNITED KINGDOM', '독일': 'GERMANY F.R', '프랑스': 'FRANCE', '이탈리아': 'ITALY',
    '네덜란드': 'NETHERLANDS', '스페인': 'SPAIN', '스위스': 'SWITZERLAND', '스웨덴': 'SWEDEN', '노르웨이': 'NORWAY',
    '덴마크': 'DENMARK', '핀란드': 'FINLAND', '벨기에': 'BELGIUM', '오스트리아': 'AUSTRIA', '폴란드': 'POLAND',
    '헝가리': 'HUNGARY', '그리스': 'GREECE', '아일랜드': 'IRELAND', '포르투갈': 'PORTUGAL', '루마니아': 'RUMANIA',
    '우크라이나': 'UKRAINE', '크로아티아': 'CROATIA', '구주기타': 'Europe Others',
    '오스트레일리아': 'AUSTRALIA', '호주': 'AUSTRALIA', '뉴질랜드': 'NEW ZEALAND', '대양주기타': 'Oceania Others',
    '터키': 'TURKEY', '이스라엘': 'ISRAEL', '아랍에미리트': 'U.A.E', '사우디아라비아': 'SAUDI ARABIA',
    '이란': 'IRAN', '중동기타': 'Middle East Others',
    '남아프리카공화국': 'SOUTH AFRICA', '이집트': 'EGYPT', '나이지리아': 'NIGERIA', '케냐': 'KENYA',
    '아프리카주기타': 'Africa Others',
    '교포': 'OVERSEAS KOREANS', '국적미상': 'STATELESS', '기타': 'OTHERS', '승무원': 'CREW',
    '마다가스카르': 'MADAGASCAR', '몬테네그로': 'MONTENEGRO', '모리셔스': 'MAURITIUS', '모로코': 'MOROCCO',
    '나미비아': 'NAMIBIA', '세르비아': 'SERBIA', '불가리아': 'BULGARIA', '키리바시': 'KIRIBATI', '피지': 'FIJI',
    '벨라루스': 'BELARUS', '온두라스': 'HONDURAS', '자메이카': 'JAMAICA', '엘살바도르': 'EL SALVADOR',
    '파라과이': 'PARAGUAY', '키르기스스탄': 'KYRGYZSTAN', '바레인': 'BAHRAIN', '바하마': 'BAHAMAS',
    '잠비아': 'ZAMBIA', '에티오피아': 'ETHIOPIA', '가나': 'GHANA', '볼리비아': 'BOLIVIA', '알바니아': 'ALBANIA',
    '베네수엘라': 'VENEZUALA', '리히텐슈타인': 'LIECHTENSTEIN', '아프가니스탄': 'AFGHANISTAN', '앙골라': 'ANGOLA',
    '부룬디': 'BURUNDI', '타지키스탄': 'TADJIKISTAN', '우루과이': 'URUGUAY', '리비아': 'LIBYA',
    '부탄': 'BHUTAN', '러시아(연방)': 'RUSSIA', '브루나이': 'BRUNEI', '라오스': 'LAOS', 
    '이라크': 'IRAQ', '예멘공화국': 'YEMEN', '카타르': 'QATAR', '벨로루시': 'BELARUS', 
    '과테말라': 'GUATEMALA', '아랍에미리트연합': 'U.A.E', '수단': 'SUDAN', '체코': 'CZECH REPUBLIC', 
    '라트비아': 'LATVIA', '미상': 'STATELESS', '튀니지': 'TUNISIA', '코트디부아르': 'COTE D IVOIRE', 
    '에스토니아': 'ESTONIA', '알제리': 'ALGERIA', '솔로몬군도': 'SOLOMON ISLANDS', '쿠웨이트': 'KUWAIT', 
    '콩고민주공화국': 'D.R.CONGO', '슬로베니아': 'SLOVENIA', '국제연합': 'U.N.', '유고슬라비아': 'YUGOSLAVIA', 
    '라이베리아': 'LIBERIA', '통가': 'TONGA', '레바논': 'LEBANON', '파나마': 'PANAMA', '레소토': 'LESOTHO', 
    '시리아': 'SYRIA', '아이티': 'HAITI', '슬로바크': 'SLOVAKIA', '키프로스': 'CYPRUS', '리투아니아': 'LITHUANIA', 
    '우간다': 'UGANDA', '모잠비크': 'MOZAMBIQUE', '바베이도스': 'BARBADOS', '몰디브': 'MALDIVES', '요르단': 'JORDAN', 
    '파푸아뉴기니': 'PAPUA NEW GUINEA', '탄자니아': 'TANZANIA', '감비아': 'GAMBIA', '몰도바': 'MOLDOVA', 
    '에콰도르': 'ECUADOR', '룩셈부르크': 'LUXEMBOURG', '마샬군도': 'MARSHALL ISLANDS', '짐바브웨': 'ZIMBABWE', 
    '말리': 'MALI', '도미니카공화국': 'DOMINICAN REPUBLIC', '미크로네시아': 'MICRONESIA', '사모아': 'SAMOA', 
    '적도기니': 'EQUATORIAL GUINEA', '팔레스타인': 'PALESTINE', '토고': 'TOGO', '티모르': 'EAST TIMOR', 
    '아이슬란드': 'ICELAND', '몰타': 'MALTA', '수리남': 'SURINAME', '기니': 'GUINEA', '투르크메니스탄': 'TURKMENISTAN', 
    '바티칸': 'VATICAN', '세인트크리스토퍼네비스': 'ST. KITTS-NEVIS', '트리니다드토바고': 'TRINIDAD AND TOBAGO', 
    '세인트루시아': 'ST. LUCIA', '보츠와나': 'BOTSWANA', '코소보': 'KOSOVO', '아르메니아': 'ARMENIA', 
    '코스타리카': 'COSTA RICA', '카메룬': 'CAMEROON', '베냉': 'BENIN', '말라위': 'MALAWI', '팔라우': 'PALAU', 
    '세이셸': 'SEYCHELLES', '기니비사우': 'GUINEA-BISSAU', '그루지야': 'GEORGIA', '쿠바': 'CUBA', 
    '마케도니아': 'MACEDONIA', '투발루': 'TUVALU', '아제르바이잔': 'AZERBAIJAN', '오만': 'OMAN', '콩고': 'CONGO', 
    '안도라': 'ANDORRA', '지부티': 'DJIBOUTI', '앤티가바부다': 'ANTIGUA AND BARBUDA', '니카라과': 'NICARAGUA', 
    '나우루': 'NAURU', '비누아투': 'VANUATU', '니제르': 'NIGER', '가봉': 'GABON', '부르키나파소': 'BURKINA FASO', 
    '도미니카연방': 'DOMINICA', '세네갈': 'SENEGAL', '세인트빈센트그레나딘': 'ST. VINCENT', '르완다': 'RWANDA', 
    '그레나다': 'GRENADA', '소말리아': 'SOMALIA', '보스니아-헤르체고비나': 'BOSNIA-HERZEGOVINA', '가이아나': 'GUYANA', 
    '모나코': 'MONACO', '모리타니': 'MAURITANIA', '벨리즈': 'BELIZE', '코모로': 'COMOROS', '카보베르데': 'CAPE VERDE', 
    '남수단공화국': 'SOUTH SUDAN', '에리트레아': 'ERITREA', '스와질란드': 'SWAZILAND', '차드': 'CHAD'
}

In [6]:
# ==========================================
# 5. Apply All Translations
# ==========================================
df_201605['Purpose'] = df_201605['Purpose'].replace(purpose_map)
df_201605['Gender'] = df_201605['Gender'].replace(gender_map)
df_201605['Age'] = df_201605['Age'].replace(age_map)
df_201605['Mode of Transport'] = df_201605['Mode of Transport'].replace(mode_map)
df_201605['Region'] = df_201605['Region'].replace(region_map)
df_201605['Transport'] = df_201605['Transport'].replace(transport_map)
df_201605['Nationality'] = df_201605['Nationality'].replace(nation_map)

In [7]:
# ==========================================
# 6. Final Check
# ==========================================
print("2016.05 Data Cleaned & Translated Successfully!")
print(df_201605.head())

2016.05 Data Cleaned & Translated Successfully!
     Month                   Purpose    Age  Gender  Nationality  \
0  2016-05                      Tour  41-50  Female        JAPAN   
1  2016-05                      Tour  51-60    Male        JAPAN   
2  2016-05                      Tour  31-40    Male        JAPAN   
3  2016-05                      Tour  31-40  Female       TAIWAN   
4  2016-05  Training of Study Abroad  21-30  Female  CHINA P. R.   

                 Region        Transport Mode of Transport  Number  
0  Asia and the Pacific             Jeju           Airport     325  
1  Asia and the Pacific           Gimhae           Airport    3102  
2  Asia and the Pacific  Incheon Airport           Airport    6296  
3  Asia and the Pacific  Incheon Airport           Airport    8594  
4  Asia and the Pacific  Incheon Airport           Airport    3503  


In [8]:
# 1. Initialize your list, but put your newly translated 2016.05 data inside it first!
li = [df_201605]

In [9]:
# 2. Get the list of all your English 2013-2021 files
# UPDATE THIS PATH to wherever you saved the 90+ English files
path = r'C:\Users\josep\Desktop\ICS\ICS 604\Python Virtual Enviroments\Sandbox 1\Korean Travel Data\*.xlsx' 
all_files = glob.glob(path)

print("Starting to load and merge files. This may take a minute...")

for filename in all_files:
    # Read the data sheet from the English file
    df = pd.read_excel(filename, sheet_name=1, header=0) 
    
    # -> THE FIX: Catch the KTO schema drift! 
    # If the column is named '\', rename it back to 'Age'. 
    # If the file is from 2013 and already has 'Age', this safely does nothing.
    df = df.rename(columns={'\\': 'Age'})
    
    # NEW: Replace the dot with a hyphen so Pandas doesn't treat it as a decimal!
    date_str = os.path.basename(filename).split(' ')[0].replace('.', '-')
    df['Month'] = date_str
    
    # Add this month's data to our list
    li.append(df)

print("All files loaded. Now concatenating into master dataframe...")

Starting to load and merge files. This may take a minute...
All files loaded. Now concatenating into master dataframe...


In [10]:
# 3. Use pd.concat to vertically stack all the DataFrames in the list together
master_df = pd.concat(li, axis=0, ignore_index=True)

In [11]:
# 4. Save the final, unified DataFrame to a CSV
# index=False ensures we don't save the row numbers as a useless column
master_df.to_csv('Master_Tourism_Data_2013_2021.csv', index=False)

print(f"Success! Merged df_201605 and {len(all_files)} English files into Master_Tourism_Data_2013_2021.csv")

Success! Merged df_201605 and 98 English files into Master_Tourism_Data_2013_2021.csv


In [ ]:
# Added encoding='utf-8-sig' to force Windows Excel to read the Korean correctly
# master_df.to_csv('Master_Tourism_Data_2013_2021.csv', index=False, encoding='utf-8-sig')

In [12]:
master_df.shape

(921812, 9)